# M03-01 — Enriquecimiento

Referencia de validación. El alumno trabaja en `notebooks/alumno/M03-01-enriquecimiento.ipynb`.


## Celda 0 — localizar el repo


In [ ]:
import sys
from pathlib import Path

_here = Path.cwd().resolve()
ROOT = next(
    p
    for p in [_here, *_here.parents]
    if (p / "labs" / "_shared" / "session.py").is_file()
)
sys.path.insert(0, str(ROOT / "labs" / "_shared"))

from paths import RAW, STAGING, CURATED
from session import get_spark

print("ROOT   ", ROOT)
print("RAW    ", RAW, "existe:", RAW.is_dir())
print("STAGING", STAGING)
print("CURATED", CURATED)


In [ ]:
from pyspark.sql.functions import col, date_format
spark = get_spark("novashop-m03")
orders = spark.read.parquet(str(STAGING / "orders_clean"))
items = spark.read.parquet(str(STAGING / "order_items_clean"))
print(orders.count(), items.count())
assert orders.count() == 788 and items.count() == 2010
lines = items.join(orders, "order_id", "inner")
print("inner", lines.count())
assert lines.count() == 1980
lines = lines.withColumn(
    "gmv_line", col("qty") * col("unit_price") * (1 - col("discount"))
).withColumn("order_month", date_format(col("order_ts"), "yyyy-MM"))
print("nulos", lines.where(col("gmv_line").isNull()).count())
print("negativos", lines.where(col("gmv_line") < 0).count())
assert lines.where(col("gmv_line").isNull()).count() == 0
assert lines.where(col("gmv_line") < 0).count() == 13
print("high_value", lines.where(col("gmv_line") >= 500).count())
assert lines.where(col("gmv_line") >= 500).count() == 506
lines.write.mode("overwrite").parquet(str(STAGING / "lines_enriched"))
print("M03-01 OK")
